# CNN Batch Size Comparison

**Model Lead follow-up.** The published CNN baseline (`cnn_baseline.keras` on Hugging Face) was trained with batch size 32. This notebook trains the *same architecture* at several different batch sizes, evaluates each on the same held-out test set, and compares all of them against the published baseline — to see whether batch size alone meaningfully changes result quality before handing anything further to tuning.

**Batch sizes tested:** 8, 16, 32 (baseline), 64, 128

| Section | What happens |
|---|---|
| 0 | Setup — clone, install, copy data |
| 1 | Rebuild dataset (same merge/preprocessing as the baseline) |
| 2 | Load the published baseline's metrics from Hugging Face, for reference |
| 3 | Train one fresh model per batch size, same architecture, same seed |
| 4 | Evaluate every model on the same test set |
| 5 | Compare — table + chart, accuracy/loss/training time vs. batch size |
| 6 | Push results to Hugging Face |

## 0. Setup

In [ ]:
!git clone https://github.com/neuroarcane/dental-cavity-detector.git
%cd dental-cavity-detector
!pip install -r requirements.txt
!nvidia-smi

In [ ]:
from pathlib import Path

data_root = Path('/content/dental_data')  # change to /kaggle/working/dental_data on Kaggle
data_root.mkdir(parents=True, exist_ok=True)
!cp -r "data/raw/Dental X-ray.v1i.yolov11" {data_root}/
!cp -r "data/raw/Dental X-Ray Panoramic Dataset" {data_root}/
(data_root / "Dental X-ray.v1i.yolov11.zip.extracted").touch()
(data_root / "Dental X-Ray Panoramic Dataset.zip.extracted").touch()

# On Kaggle only — its base image falsely triggers Colab detection. Skip this on real Colab.
# import src.data.prepare_dataset as prepare_dataset
# prepare_dataset.is_colab = lambda: False

In [ ]:
from src.data.prepare_dataset import merge_datasets, print_merge_summary
from src.data.split import remove_exact_duplicates, stratified_resplit
from src.data.balance import oversample_minority_classes

stats = merge_datasets()
print_merge_summary(stats)
remove_exact_duplicates()
stratified_resplit(train_frac=0.7, valid_frac=0.15, test_frac=0.15)
oversample_minority_classes(split='train', target_ratio=0.5, max_duplicates_per_image=5)

!cat data/processed/data.yaml

## 1. Rebuild the dataset pipeline

Same single-label conversion as the published baseline — identical class list, identical priority rule, so results are directly comparable.

In [ ]:
import sys, yaml, json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR = Path('data/processed')
IMG_SIZE = 128
SEED = 42

with open(DATA_DIR / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)
yaml_class_names = data_yaml['names']

CLASS_NAMES = ['Cavity', 'Filling', 'Crown', 'Impacted Tooth']
PRIORITY = ['Cavity', 'Crown', 'Impacted Tooth', 'Filling']
NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
def build_singlelabel_index(split_dir: Path) -> pd.DataFrame:
    images_dir, labels_dir = split_dir / 'images', split_dir / 'labels'
    rows = []
    for img_path in sorted(images_dir.glob('*.*')):
        label_path = labels_dir / f'{img_path.stem}.txt'
        present = set()
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    present.add(yaml_class_names[int(line.split()[0])])
        chosen = PRIORITY[-1]
        for cls in PRIORITY:
            if cls in present:
                chosen = cls
                break
        rows.append({'path': str(img_path), 'class_id': CLASS_NAMES.index(chosen)})
    return pd.DataFrame(rows)

train_df = build_singlelabel_index(DATA_DIR / 'train')
val_df   = build_singlelabel_index(DATA_DIR / 'valid')
test_df  = build_singlelabel_index(DATA_DIR / 'test')
print('train:', len(train_df), ' val:', len(val_df), ' test:', len(test_df))

In [ ]:
def make_dataset(df: pd.DataFrame, batch_size: int, shuffle: bool = False):
    paths, labels = df['path'].values, df['class_id'].values
    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

present_classes = np.unique(train_df['class_id'].values)
weights = compute_class_weight(class_weight='balanced', classes=present_classes, y=train_df['class_id'].values)
class_weight = {i: 1.0 for i in range(NUM_CLASSES)}
class_weight.update(dict(zip(present_classes, weights)))
print('Class weights:', {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

## 2. Pull the published baseline's metrics for reference

So the comparison table includes the actual published number, not a rerun that might drift slightly from it.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi, login
from google.colab import userdata  # swap for kaggle_secrets.UserSecretsClient on Kaggle

login(token=userdata.get('HF_TOKEN'))
HF_REPO_ID = 'aparnamohankumar/dental-cavity-detector'
api = HfApi()

baseline_summary_path = hf_hub_download(repo_id=HF_REPO_ID, filename='cnn_baseline_summary.json')
with open(baseline_summary_path) as f:
    baseline_summary = json.load(f)
print('Published baseline:', baseline_summary)

## 3. Train one model per batch size

Identical architecture and epoch count to the published baseline — only `batch_size` changes between runs, so any difference in results is attributable to batch size, not something else changing at the same time.

In [ ]:
def build_cnn_baseline(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

In [ ]:
BATCH_SIZES = [8, 16, 32, 64, 128]
EPOCHS = 15  # kept modest so all 5 runs finish in a reasonable time; same epoch count for every run

results = {}
histories = {}

for bs in BATCH_SIZES:
    print(f'\n=== Training with batch size {bs} ===')
    tf.random.set_seed(SEED)  # reset seed each run so only batch size differs

    train_ds = make_dataset(train_df, batch_size=bs, shuffle=True)
    val_ds   = make_dataset(val_df, batch_size=bs)
    test_ds  = make_dataset(test_df, batch_size=bs)

    model = build_cnn_baseline()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    import time
    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, verbose=0)
    train_time = time.time() - start

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)

    results[bs] = {
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'train_time_sec': round(train_time, 1),
        'final_train_acc': float(history.history['accuracy'][-1]),
        'final_val_acc': float(history.history['val_accuracy'][-1]),
    }
    histories[bs] = history.history
    print(f'batch={bs}: test_acc={test_acc:.4f}, test_loss={test_loss:.4f}, time={train_time:.1f}s')

    del model
    tf.keras.backend.clear_session()

## 4. Results table

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = 'batch_size'
results_df = results_df.reset_index()

# add the published baseline as its own row for direct comparison
baseline_row = pd.DataFrame([{
    'batch_size': '32 (published baseline)',
    'test_accuracy': baseline_summary.get('test_accuracy', None),
    'test_loss': baseline_summary.get('test_loss', None),
    'train_time_sec': None,
    'final_train_acc': None,
    'final_val_acc': None,
}])

comparison_df = pd.concat([results_df, baseline_row], ignore_index=True)
comparison_df

## 5. Compare visually

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(results_df['batch_size'], results_df['test_accuracy'], 'o-')
axes[0].axhline(baseline_summary.get('test_accuracy', 0), color='gray', linestyle='--', label='published baseline (bs=32)')
axes[0].set_title('Test Accuracy vs Batch Size'); axes[0].set_xlabel('batch size'); axes[0].legend()

axes[1].plot(results_df['batch_size'], results_df['test_loss'], 'o-', color='indianred')
axes[1].axhline(baseline_summary.get('test_loss', 0), color='gray', linestyle='--', label='published baseline (bs=32)')
axes[1].set_title('Test Loss vs Batch Size'); axes[1].set_xlabel('batch size'); axes[1].legend()

axes[2].bar(results_df['batch_size'].astype(str), results_df['train_time_sec'], color='seagreen')
axes[2].set_title('Training Time vs Batch Size'); axes[2].set_xlabel('batch size'); axes[2].set_ylabel('seconds')

plt.tight_layout()
plt.savefig('cnn_batch_size_comparison.png', dpi=150)
plt.show()

In [ ]:
# Training curves overlay — val accuracy per batch size, same epoch count
fig, ax = plt.subplots(figsize=(8, 5))
for bs, hist in histories.items():
    ax.plot(hist['val_accuracy'], label=f'batch={bs}')
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy'); ax.set_title('Validation Accuracy by Batch Size')
ax.legend()
plt.tight_layout()
plt.savefig('cnn_batch_size_val_curves.png', dpi=150)
plt.show()

## 6. Push results to Hugging Face

In [ ]:
comparison_df.to_csv('cnn_batch_size_comparison.csv', index=False)

api.upload_file(path_or_fileobj='cnn_batch_size_comparison.csv', path_in_repo='cnn_batch_size_comparison.csv', repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj='cnn_batch_size_comparison.png', path_in_repo='cnn_batch_size_comparison.png', repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj='cnn_batch_size_val_curves.png', path_in_repo='cnn_batch_size_val_curves.png', repo_id=HF_REPO_ID)

print('Pushed comparison CSV + 2 charts to Hugging Face.')

## Notes for the team

- All runs use the **same architecture, seed, epoch count, and data** — batch size is the only variable changed, so differences in the table above are attributable to batch size specifically.
- Very small batch sizes (8) mean noisier gradient updates per step but more updates per epoch; very large batch sizes (128) mean smoother updates but fewer of them — check the val-accuracy curve chart to see whether any batch size converged meaningfully faster or slower, not just the final number.
- If the differences here are small, that's a legitimate finding too — it means batch size isn't a lever worth spending tuning time on, and Temirlan can focus tuning effort elsewhere (e.g. learning rate, architecture depth, or specifically improving Cavity recall).